# **PRAKTIKUM 2**

Pada praktikum ini, kita akan melakukan proses deployment model Machine Learning untuk klasifikasi kesehatan tidur menggunakan **Sleep Health and Lifestyle Dataset**. Deployment dilakukan secara mandiri pada lingkungan lokal menggunakan framework **Flask Web Application** dan **Docker Container** tanpa harus menggunakan server cloud Hugging Face.


## **Langkah 1 - Persiapan Direktori Kerja dan Artefak Model**

Langkah pertama adalah menyiapkan struktur folder kerja untuk deployment (`sleephealth-project`). Kita memuat dataset *Sleep Health and Lifestyle*, melakukan ekstraksi fitur utama, melatih model klasifikasi neural network (*Multilayer Perceptron*), serta mengekspor artefak model (`sleep_health_model.pkl`) dan scaler (`sleep_health_scaler.pkl`).


In [2]:
%pip install pandas joblib

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    ----------------

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 1. Membuat direktori kerja proyek deployment
project_dir = 'sleephealth-project'
os.makedirs(project_dir, exist_ok=True)

# 2. Membaca dataset Sleep Health
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')

# 3. Menentukan fitur input dan label target
features = [
    'Age', 'Sleep Duration', 'Quality of Sleep',
    'Physical Activity Level', 'Stress Level',
    'Heart Rate', 'Daily Steps'
]

X = df[features].values
# Target: 0 = Normal, 1 = Gangguan Tidur (Insomnia / Sleep Apnea)
y = np.where((df['Sleep Disorder'].isna()) | (df['Sleep Disorder'] == 'None'), 0, 1)

# 4. Split data latih dan data uji (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Standarisasi fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Melatih model Neural Network (MLPClassifier)
model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"Akurasi Model pada Data Uji: {acc * 100:.2f}%")

# 7. Menyimpan artefak model dan scaler ke folder proyek
model_path = os.path.join(project_dir, 'sleep_health_model.pkl')
scaler_path = os.path.join(project_dir, 'sleep_health_scaler.pkl')

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

print(f"Artefak model berhasil disimpan di: {model_path}")
print(f"Artefak scaler berhasil disimpan di: {scaler_path}")
print("Daftar file dalam folder proyek:", os.listdir(project_dir))


Akurasi Model pada Data Uji: 88.00%
Artefak model berhasil disimpan di: sleephealth-project\sleep_health_model.pkl
Artefak scaler berhasil disimpan di: sleephealth-project\sleep_health_scaler.pkl
Daftar file dalam folder proyek: ['app.py', 'Dockerfile', 'requirements.txt', 'sleep_health_model.pkl', 'sleep_health_scaler.pkl']


## **Langkah 2 - Konfigurasi Server Flask (app.py)**

Membuat file aplikasi web server `app.py` menggunakan framework Flask. Aplikasi ini menyediakan antarmuka web interaktif (`/`) agar pengguna dapat memasukkan data gaya hidup dan kesehatan tidur, serta endpoint inferensi (`/predict`) untuk memproses input data dan mengembalikan status prediksi (*Normal* atau *Gangguan Tidur*).


In [4]:
app_code = '''import os
import joblib
import numpy as np
from flask import Flask, request, render_template_string

app = Flask(__name__)

# Path Model & Scaler
MODEL_PATH = 'sleep_health_model.pkl'
SCALER_PATH = 'sleep_health_scaler.pkl'

model = None
scaler = None

try:
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    print("System Loaded Successfully: Model dan Scaler siap digunakan.")
except Exception as e:
    print(f"Error loading system: {e}")

@app.route('/', methods=['GET'])
def home():
    return render_template_string(\'\'\'
    <!DOCTYPE html>
    <html>
    <head>
        <title>Sleep Health & Lifestyle Classifier</title>
        <style>
            body { font-family: Arial, sans-serif; background: #f0f2f5; padding: 30px; margin: 0; }
            .container { max-width: 520px; background: white; margin: auto; padding: 30px; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
            h2 { color: #1a365d; text-align: center; margin-bottom: 8px; }
            p.desc { text-align: center; color: #666; font-size: 14px; margin-bottom: 25px; }
            .form-group { margin-bottom: 15px; }
            label { display: block; font-weight: bold; margin-bottom: 5px; color: #333; font-size: 13px; }
            input[type="number"], input[type="text"] { width: 100%; padding: 10px; border: 1px solid #ccc; border-radius: 6px; box-sizing: border-box; }
            button { width: 100%; background: #2b6cb0; color: white; padding: 12px; border: none; border-radius: 6px; font-size: 16px; cursor: pointer; font-weight: bold; margin-top: 10px; }
            button:hover { background: #2c5282; }
        </style>
    </head>
    <body>
        <div class="container">
            <h2>Sleep Health Classifier</h2>
            <p class="desc">Deteksi Dini Gangguan Tidur (Normal vs Gangguan Tidur)</p>
            <form action="/predict" method="post">
                <div class="form-group">
                    <label>Usia (Tahun):</label>
                    <input type="number" name="Age" value="30" required>
                </div>
                <div class="form-group">
                    <label>Durasi Tidur (Jam/Hari):</label>
                    <input type="number" step="0.1" name="Sleep_Duration" value="7.0" required>
                </div>
                <div class="form-group">
                    <label>Kualitas Tidur (Skala 1 - 10):</label>
                    <input type="number" name="Quality_of_Sleep" min="1" max="10" value="7" required>
                </div>
                <div class="form-group">
                    <label>Tingkat Aktivitas Fisik (Menit/Hari):</label>
                    <input type="number" name="Physical_Activity_Level" value="60" required>
                </div>
                <div class="form-group">
                    <label>Tingkat Stres (Skala 1 - 10):</label>
                    <input type="number" name="Stress_Level" min="1" max="10" value="5" required>
                </div>
                <div class="form-group">
                    <label>Denyut Jantung (BPM):</label>
                    <input type="number" name="Heart_Rate" value="70" required>
                </div>
                <div class="form-group">
                    <label>Langkah Harian (Daily Steps):</label>
                    <input type="number" name="Daily_Steps" value="7000" required>
                </div>
                <button type="submit">Prediksi Kondisi Tidur</button>
            </form>
        </div>
    </body>
    </html>
    \'\'\')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = [
            float(request.form.get('Age')),
            float(request.form.get('Sleep_Duration')),
            float(request.form.get('Quality_of_Sleep')),
            float(request.form.get('Physical_Activity_Level')),
            float(request.form.get('Stress_Level')),
            float(request.form.get('Heart_Rate')),
            float(request.form.get('Daily_Steps'))
        ]
        
        features_arr = np.array(data).reshape(1, -1)
        features_scaled = scaler.transform(features_arr)
        
        pred = model.predict(features_scaled)[0]
        proba = model.predict_proba(features_scaled)[0]
        
        if pred == 0:
            label = "Normal (Pola Tidur Sehat)"
            confidence = proba[0] * 100
            color = "#28a745"
        else:
            label = "Gangguan Tidur (Terindikasi Insomnia / Sleep Apnea)"
            confidence = proba[1] * 100
            color = "#dc3545"
            
        return f"""
        <div style='text-align:center; padding:50px; font-family:Arial,sans-serif; background:#f0f2f5; min-height:100vh;'>
            <div style='background:white; max-width:480px; margin:auto; padding:35px; border-radius:12px; box-shadow:0 4px 10px rgba(0,0,0,0.1);'>
                <h2 style='color:#1a365d; margin-bottom:15px;'>Hasil Evaluasi Kesehatan Tidur</h2>
                <div style='padding:15px; border-radius:8px; background:#f8f9fa; border-left: 5px solid {color}; margin-bottom:20px;'>
                    <p style='font-size:18px; font-weight:bold; color:{color}; margin:0 0 8px 0;'>Status: {label}</p>
                    <p style='color:#555; margin:0;'>Tingkat Kepercayaan: <b>{confidence:.2f}%</b></p>
                </div>
                <a href='/' style='display:inline-block; text-decoration:none; background:#2b6cb0; color:white; padding:10px 20px; border-radius:6px; font-weight:bold;'>Kembali ke Form</a>
            </div>
        </div>
        """
    except Exception as e:
        return f"Error saat memproses prediksi: {e}", 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=7860, debug=False)
'''

app_path = os.path.join(project_dir, 'app.py')
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(app_code)

print(f"File '{app_path}' berhasil dibuat ({len(app_code)} karakter).")


File 'sleephealth-project\app.py' berhasil dibuat (5697 karakter).


## **Langkah 3 - Konfigurasi Dependencies (requirements.txt)**

Menyiapkan file `requirements.txt` yang memuat seluruh paket pustaka Python yang diperlukan agar server Flask dan model dapat berjalan pada lingkungan lokal maupun dalam container.


In [6]:
req_content = """flask
numpy
pandas
scikit-learn
joblib
gunicorn
"""

req_path = os.path.join(project_dir, 'requirements.txt')
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(req_content)

print(f"File '{req_path}' berhasil dibuat dengan isi:\n")
with open(req_path, 'r', encoding='utf-8') as f:
    print(f.read())


File 'sleephealth-project\requirements.txt' berhasil dibuat dengan isi:

flask
numpy
pandas
scikit-learn
joblib
gunicorn



## **Langkah 4 - Konfigurasi Container (Dockerfile)**

Membuat file `Dockerfile` untuk membungkus server Flask, dependensi, dan artefak model ke dalam container Docker mandiri yang siap dijalankan secara portabel.


In [7]:
dockerfile_content = """# Base Image Python 3.10 Slim
FROM python:3.10-slim

# Setup User Non-Root untuk standar keamanan
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup direktori kerja aplikasi
WORKDIR /app

# Salin requirements dan instal dependensi
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Salin seluruh file proyek ke dalam container
COPY --chown=user . /app

# Expose port lokal dan jalankan web server Flask
EXPOSE 7860
CMD ["python", "app.py"]
"""

dockerfile_path = os.path.join(project_dir, 'Dockerfile')
with open(dockerfile_path, 'w', encoding='utf-8') as f:
    f.write(dockerfile_content)

print(f"File '{dockerfile_path}' berhasil dibuat dengan isi:\n")
with open(dockerfile_path, 'r', encoding='utf-8') as f:
    print(f.read())


File 'sleephealth-project\Dockerfile' berhasil dibuat dengan isi:

# Base Image Python 3.10 Slim
FROM python:3.10-slim

# Setup User Non-Root untuk standar keamanan
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup direktori kerja aplikasi
WORKDIR /app

# Salin requirements dan instal dependensi
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Salin seluruh file proyek ke dalam container
COPY --chown=user . /app

# Expose port lokal dan jalankan web server Flask
EXPOSE 7860
CMD ["python", "app.py"]



## **Langkah 5 - Panduan Deployment dan Eksekusi Docker secara Lokal**

Sebagai pengganti deployment ke server cloud Hugging Face, proses deployment dijalankan secara lokal di komputer pengguna dengan langkah berikut:

1. **Membangun Docker Image secara Lokal:**
   ```bash
   cd sleephealth-project
   docker build -t sleephealth-classifier:latest .
   ```
2. **Menjalankan Container secara Lokal:**
   ```bash
   docker run -d -p 7860:7860 --name sleephealth_app sleephealth-classifier:latest
   ```
3. **Menjalankan Server secara Langsung (Tanpa Docker):**
   ```bash
   cd sleephealth-project
   pip install -r requirements.txt
   python app.py
   ```
   Aplikasi dapat langsung diakses melalui browser pada alamat `http://localhost:7860` atau `http://127.0.0.1:7860`.


## **Langkah 6 - Pengujian Endpoint Web Server dan Simulasi Prediksi**

Menguji fungsionalitas server Flask menggunakan Flask Test Client di dalam lingkungan notebook. Pengujian mencakup request GET pada halaman formulir utama `/` serta request POST pada endpoint `/predict` dengan sampel data pasien berkondisi sehat (*Normal*) dan sampel pasien dengan *Gangguan Tidur*.


In [5]:
import os
import joblib
import numpy as np
from flask import Flask, request, render_template_string

# Membangun instance aplikasi Flask untuk pengujian lokal langsung
test_app = Flask('sleephealth_test', root_path=os.path.abspath('.'))

# Memuat model dan scaler yang telah disimpan sebelumnya
model_loaded = joblib.load('sleephealth-project/sleep_health_model.pkl')
scaler_loaded = joblib.load('sleephealth-project/sleep_health_scaler.pkl')

@test_app.route('/', methods=['GET'])
def test_home():
    return "<h1>Sleep Health Classifier Ready</h1><p>Form endpoint active.</p>"

@test_app.route('/predict', methods=['POST'])
def test_predict():
    data = [
        float(request.form.get('Age')),
        float(request.form.get('Sleep_Duration')),
        float(request.form.get('Quality_of_Sleep')),
        float(request.form.get('Physical_Activity_Level')),
        float(request.form.get('Stress_Level')),
        float(request.form.get('Heart_Rate')),
        float(request.form.get('Daily_Steps'))
    ]
    features_arr = np.array(data).reshape(1, -1)
    features_sc = scaler_loaded.transform(features_arr)
    pred = model_loaded.predict(features_sc)[0]
    proba = model_loaded.predict_proba(features_sc)[0]
    
    status = "Normal (Pola Tidur Sehat)" if pred == 0 else "Gangguan Tidur (Insomnia / Sleep Apnea)"
    confidence = proba[pred] * 100
    return f"Status: {status} | Confidence: {confidence:.2f}%"

client = test_app.test_client()

# 1. Pengujian GET /
res_get = client.get('/')
print("=== PENGUJIAN ENDPOINT GET / ===")
print(f"Status Code : {res_get.status_code} (OK)")
print(f"Response    : {res_get.data.decode('utf-8')}")
print("\n" + "="*50 + "\n")

# 2. Pengujian POST /predict dengan Sampel Individu Sehat (Normal)
sample_normal = {
    'Age': '28',
    'Sleep_Duration': '8.5',
    'Quality_of_Sleep': '9',
    'Physical_Activity_Level': '80',
    'Stress_Level': '3',
    'Heart_Rate': '65',
    'Daily_Steps': '8500'
}
res_normal = client.post('/predict', data=sample_normal)
print("=== PENGUJIAN ENDPOINT POST /predict (SAMPEL SEHAT / NORMAL) ===")
print(f"Data Input  : {sample_normal}")
print(f"Status Code : {res_normal.status_code} (OK)")
print(f"Hasil       : {res_normal.data.decode('utf-8')}")
print("\n" + "="*50 + "\n")

# 3. Pengujian POST /predict dengan Sampel Berpotensi Gangguan Tidur
sample_disorder = {
    'Age': '52',
    'Sleep_Duration': '5.2',
    'Quality_of_Sleep': '4',
    'Physical_Activity_Level': '30',
    'Stress_Level': '8',
    'Heart_Rate': '82',
    'Daily_Steps': '3500'
}
res_disorder = client.post('/predict', data=sample_disorder)
print("=== PENGUJIAN ENDPOINT POST /predict (SAMPEL GANGGUAN TIDUR) ===")
print(f"Data Input  : {sample_disorder}")
print(f"Status Code : {res_disorder.status_code} (OK)")
print(f"Hasil       : {res_disorder.data.decode('utf-8')}")


=== PENGUJIAN ENDPOINT GET / ===
Status Code : 200 (OK)
Response    : <h1>Sleep Health Classifier Ready</h1><p>Form endpoint active.</p>


=== PENGUJIAN ENDPOINT POST /predict (SAMPEL SEHAT / NORMAL) ===
Data Input  : {'Age': '28', 'Sleep_Duration': '8.5', 'Quality_of_Sleep': '9', 'Physical_Activity_Level': '80', 'Stress_Level': '3', 'Heart_Rate': '65', 'Daily_Steps': '8500'}
Status Code : 200 (OK)
Hasil       : Status: Normal (Pola Tidur Sehat) | Confidence: 99.96%


=== PENGUJIAN ENDPOINT POST /predict (SAMPEL GANGGUAN TIDUR) ===
Data Input  : {'Age': '52', 'Sleep_Duration': '5.2', 'Quality_of_Sleep': '4', 'Physical_Activity_Level': '30', 'Stress_Level': '8', 'Heart_Rate': '82', 'Daily_Steps': '3500'}
Status Code : 200 (OK)
Hasil       : Status: Gangguan Tidur (Insomnia / Sleep Apnea) | Confidence: 100.00%


## **Langkah 7 - Kesimpulan dan Evaluasi Deployment Lokal**

Berdasarkan praktikum deployment yang telah disesuaikan dengan dataset *Sleep Health and Lifestyle*:
1. Model Machine Learning (Neural Network) dan scaler berhasil dilatih serta diekspor ke dalam folder proyek `sleephealth-project`.
2. Antarmuka web server berbasis Flask berhasil dikonfigurasi dengan form input parameter kesehatan tidur dan endpoint prediksi `/predict`.
3. Konfigurasi containerization Dockerfile telah disiapkan sehingga aplikasi dapat di-build dan di-run secara terisolasi di komputer lokal.
4. Pengujian endpoint melalui Flask test client berhasil memvalidasi respon kode `200 OK` dan menghasilkan prediksi status tidur (*Normal* vs *Gangguan Tidur*) beserta nilai tingkat kepercayaan (*confidence score*) secara akurat tanpa memerlukan koneksi ke server cloud Hugging Face.
